# zenpy-garden core

> Builder API for creating Zendesk Garden FastHTML components

This module provides the factory functions and CSS headers needed to create FastHTML components that wrap [Zendesk Garden CSS](https://github.com/zendeskgarden/css-components). The component factory pattern is adapted from [fhdaisy](https://github.com/answerdotai/fhdaisy) by AnswerDotAI.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
import fasthtml.components as fh
from fasthtml.jupyter import *

import inspect

## Component Factory

In [ ]:
#| export
def clstoname(
    compcls: str  # Garden CSS class name (e.g. 'c-btn')
    ) -> str:      # PascalCase component name (e.g. 'Btn')
    "Convert a Garden CSS class name to a PascalCase component name"
    assert compcls[:2] == 'c-'
    return ''.join(s.title() for s in compcls[2:].split('-'))

def mk_compfn(
    compcls,  # Base CSS class (e.g. 'c-btn')
    tag=None, # HTML tag function to use (defaults to name)
    name=None, # Component function name (defaults to clstoname(compcls))
    xcls='',  # Extra classes to always include
    **compkw  # Additional kwargs passed to every component call
    ):
    "Create a FastHTML component function for a Garden CSS class"
    if not name: name = clstoname(compcls)
    if not tag: tag = name
    compfunc = getattr(fh, tag)

    def fn(*c, cls='', **kw):
        cls = ' '.join(f'{compcls if x[:2]=="--" else ""}{x}' for x in cls.split())
        return compfunc(*c, cls=f'{compcls} {cls} {xcls}', **compkw, **kw)

    fn.__name__ = name
    inspect.currentframe().f_back.f_globals[name] = fn


In [ ]:
mk_compfn('c-btn')
pill = Btn('.c-btn--pill', cls='--pill')
print(pill)

## Preview Helper

In [ ]:
#| export
# Core packages
bedrock_css = Link(href='https://cdn.jsdelivr.net/npm/@zendeskgarden/css-bedrock/dist/index.css', rel='stylesheet')
variables_css = Link(href='https://cdn.jsdelivr.net/npm/@zendeskgarden/css-variables/dist/index.css', rel='stylesheet')

# Component packages
buttons_css = Link(href='https://cdn.jsdelivr.net/npm/@zendeskgarden/css-buttons/dist/index.css', rel='stylesheet')
forms_css = Link(href='https://cdn.jsdelivr.net/npm/@zendeskgarden/css-forms/dist/index.css', rel='stylesheet')
tags_css = Link(href='https://cdn.jsdelivr.net/npm/@zendeskgarden/css-tags/dist/index.css', rel='stylesheet')
avatars_css = Link(href='https://cdn.jsdelivr.net/npm/@zendeskgarden/css-avatars/dist/index.css', rel='stylesheet')
anchors_css = Link(href='https://cdn.jsdelivr.net/npm/@zendeskgarden/css-anchors/dist/index.css', rel='stylesheet')

garden_hdrs = (bedrock_css, variables_css, buttons_css, forms_css, tags_css, avatars_css, anchors_css)

def mk_previewer(
    app=None,  # FastHTML app to use (creates one with garden_hdrs if None)
    cls=''     # CSS classes for the preview container div
    ):
    "Create a preview function for rendering Garden components in notebooks"
    xcls = cls
    if not app: app=FastHTML(hdrs=garden_hdrs)
    def p(*c, cls='', **kw):
        return HTMX(Div(cls=f'{xcls} {cls}')(*c), app=app, host=None, port=None, **kw)
    return p

In [ ]:
p = mk_previewer()
p(pill)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()